In [3]:
!sudo apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (1,391 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package poppler-utils.
(Reading database ... 11821

In [4]:
!pip install PyMuPDF pdf2image dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.1 MB/s eta 0:00:00


In [5]:
!pip install pymongo

In [6]:
import json
import os
import pdf2image
from PIL import Image
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import time
import asyncio
from dotenv import load_dotenv

# PDF parsing by using PDF2Image and GoogleGenAI

# RUN IT IN COLAB

This is a useful way when there are some important informations highlighted

 -> No possibility to use OCR
  
  -> Multimodal large language model as plug and play

## 1) Config variables and API Key

In [7]:
# config global static variables

PDF_PATH = "/content/Domande Chirurgia esame.pdf"
OUTPUT_PNG_FOLDER = "output_folder"
CONTENT_SECTIONS = [
    {
        "section": "chirurgia",
        "code": "chir",
        "from_page": 1,
        "to_page": 67
    }
]
JSON_OUTPUT_PATH = "/content/full_extraction.json"
JSON_METADATA_OUTPUT_PATH = "/content/metadata_full_extraction.json"

In [8]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
MONGO_DB_URI = userdata.get('MONGODB_URI')

## 2) Extract PNG single-page image from PDF

In [ ]:
def save_pdf_as_images(pdf_path, output_folder):

  images = pdf2image.convert_from_path(pdf_path)

  if not os.path.exists(output_folder):
      os.makedirs(output_folder)

  saved_png=0

  for i,image in enumerate(images, start=1):
    output_path = f"{output_folder}/output{i}.png"
    image.save(output_path, "PNG")
    saved_png+=1
    #print(f"Image saved as PNG: {output_path}")

  print(f"Saved {saved_png} images.")
  return saved_png

In [ ]:
saved_png = save_pdf_as_images(PDF_PATH, OUTPUT_PNG_FOLDER)

Saved 67 images.


## 3) Use a Vision Language Model to extract the content

In [ ]:
# config structured output class

class DomandaRisposta(BaseModel):
    num_domanda: int = Field(..., description="Numero della domanda così come riportato nell'immagine")
    domanda: str = Field(..., description="Testo completo della domanda")
    opzioni: list[str] = Field(..., description="Lista di tutte le opzioni possibili, inclusa la risposta corretta")
    risposta_corretta: str = Field(
        ..., description="Risposta evidenziata in giallo o NO_ANSWER"
    )

# config system prompt
SYSTEM_PROMPT = """
    Il seguente file contiene una serie di domande e risposte a scelta multipla. Le risposte esatte possono essere: evidenziate in giallo, oppure scritte in grassetto.
    Restituisci in output un JSON con questa struttura:
    [
        {
        "num_domanda": Numero della domanda così come riportato nell'immagine,
        "domanda": Testo della domanda completa,
        "opzioni": ["Prima opzione riportata", "Seconda opzione", eccetera]
        "risposta_corretta": La risposta evidenziata in giallo riportata fedelmente, deve essere lo stesso testo di una delle opzioni della lista
        }
    ]

    Casi possibili:
    - Caso normale (corrisponde alla maggior parte dei casi): lista di domande con risposta esatta evidenziata in giallo oppure con risposta esatta in grassetto
    - Più risposte a scelta multipla ma nessuna risposta evidenziata: risposta_corretta = "NO_ANSWER"
    - Solo la risposta corretta è presente: la risposta corretta, unica presente, è evidenziata in giallo. In questi casi, "opzioni" include nella lista solo la risposta corretta e "risposta_corretta" contiene lo stesso testo
    - Opzioni presenti ma nessuna domanda presente (può succedere a inizio pagina): in questo caso riempi "num_domanda" con -1 e "domanda" con "NO_QUESTION", riportando le opzioni tra "opzioni" se presente evidenziata in giallo

    Esempio concreto di input-output con quello che troverai nell'immagine:
        INPUT
        1. L'energia dei raggi x è direttamente proporzionale:
            - alla loro lunghezza d'onda
            - alla loro frequenza (evidenziata in giallo nell'immagine)
            - alla velocità della luce
            - alla loro elasticità
        2. L'imaging plate è:
            - un detettore che registra sulla propria superficie l'energia dei fotoni x (evidenziato in giallo nell'immagine)
            - un tipo di tubo radiogeno
            - un algoritmo di ricostruzione delle immagini in tac
            - un sistema integrato di appiattimento delle immagini
        3. I raggi X persistono nella sala radiologica, una volta interrotta la esposizione radiante, per:
            Scompaiono immediatamente (evidenziata in giallo)
        4. Per cosa non viene utilizzata la via intradermica?
            - Prova di reazione alla tubercolina
            - Test cutanei
            - Desensibilizzazione
            - Terapia immunosoppressiva

        OUTPUT
        [
            {
            "num_domanda": -1,
            "domanda": "NO_QUESTION",
            "opzioni": [
                "verificare l'identità dell'assistito in modo attivo attraverso il braccialetto identificativo e il braccialetto con codice colore della trasfusione",
                "verificare l'identità dell'assistito in modo attivo, controllare che il numero della cartella clinica corrisponda a quello del braccialetto identificativo",
                "verificare l'identità dell'assistito in modo attivo e controllare che in cartella clinica sia presente la richiesta di trasfusione"
            ],
            "risposta_corretta": "NO_ANSWER"
            },
            {
            "num_domanda": 1,
            "domanda": "L'energia dei raggi x è direttamente proporzionale:",
            "opzioni": [
                "alla loro lunghezza d'onda",
                "alla loro frequenza",
                "alla velocità della luce",
                "alla loro elasticità"
            ],
            "risposta_corretta": "alla loro frequenza"
            },
            {
            "num_domanda": 2,
            "domanda": "L'imaging plate è:",
            "opzioni": [
                "un detettore che registra sulla propria superficie l'energia dei fotoni x",
                "un tipo di tubo radiogeno",
                "un algoritmo di ricostruzione delle immagini in tac",
                "un sistema integrato di appiattimento delle immagini"
            ],
            "risposta_corretta": "un detettore che registra sulla propria superficie l'energia dei fotoni x"
            },
            {
            "num_domanda": 3,
            "domanda": "I raggi X persistono nella sala radiologica, una volta interrotta la esposizione radiante, per:",
            "opzioni": [
                "Scompaiono immediatamente"
            ],
            "risposta_corretta": "Scompaiono immediatamente"
            },
            {
            "num_domanda": 4,
            "domanda": "Per cosa non viene utilizzata la via intradermica?",
            "opzioni": [
                "Prova di reazione alla tubercolina",
                "Test cutanei",
                "Desensibilizzazione",
                "Terapia immunosoppressiva"
            ],
            "risposta_corretta": "NO_ANSWER"
            }
        ]
  """

### 3.1) Use Gemini (plug-and-play)

In [ ]:
# config client
CLIENT = genai.Client(api_key=GOOGLE_API_KEY)

# config models (list with decreasing quality)

GEMINI_MODELS = [
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
]

MIL_TOKEN=1000000
PRICING = {
    "gemini-3-flash-preview":{
        "input":0.5/MIL_TOKEN,
        "output":3/MIL_TOKEN
    },
    "gemini-2.5-flash":{
        "input":0.3/MIL_TOKEN,
        "output":2.5/MIL_TOKEN
    },
    "gemini-2.5-flash-lite":{
        "input":0.1/MIL_TOKEN,
        "output":0.4/MIL_TOKEN
    },
    "gemini-2.0-flash":{
        "input":0.1/MIL_TOKEN,
        "output":0.4/MIL_TOKEN
    }
}

In [ ]:
def is_resource_exhausted(error: Exception) -> bool:
    error_str = str(error).lower()
    return any(k in error_str for k in [
        "resourceexhausted",
        "quota exceeded",
        "rate limit",
        "429"
    ])

def generate_with_fallback(uploaded_file):
    last_exception = None

    for model_name in GEMINI_MODELS:
        try:
          start = time.time()
          response = CLIENT.models.generate_content(
              model=model_name,
              contents=[uploaded_file],
              config={
                  "response_mime_type": "application/json",
                  "response_schema": list[DomandaRisposta],
                  "temperature": 0,
                  "system_instruction": SYSTEM_PROMPT,
                  "thinking_config": {
                      "include_thoughts": False
                  }
              }
          )
          end = time.time()

          try:
            input_cost = response.usage_metadata.prompt_token_count*PRICING[model_name]["input"]
            if response.usage_metadata.thoughts_token_count:
              output_cost = (response.usage_metadata.candidates_token_count + response.usage_metadata.thoughts_token_count)*PRICING[model_name]["output"]
            else:
              output_cost = (response.usage_metadata.candidates_token_count)*PRICING[model_name]["output"]

            response_metadata = {
                "time": end-start,
                "input_cost": input_cost,
                "output_cost": output_cost
            }
          except Exception as e:
            print(f"Error getting metadata: {e}")
            response_metadata = {
                "time": end-start,
                "input_cost": 0,
                "output_cost": 0
            }
          finally:
            # sleep in caso di successo
            time.sleep(15)
            return response, response_metadata

        except Exception as e:
            last_exception = e

            if not is_resource_exhausted(e):
                raise e  # errore non gestibile → fail immediato

            print(f"Resource exhausted with {model_name}, trying next model...")
            # time.sleep(15)

    # se arriviamo qui, tutti i modelli hanno fallito
    raise RuntimeError("All Gemini models exhausted") from last_exception

In [ ]:
answers = []
response_metadata_list = []
start_index = 1

for i in range(1,saved_png + 1):
  print("Working with image:", i)

  image_path = f"{OUTPUT_PNG_FOLDER}/output{i}.png"

  # upload immagine a Gemini
  uploaded_file = CLIENT.files.upload(file=image_path)

  response, response_metadata = generate_with_fallback(uploaded_file)
  raw_text = response.text

  print("Answer generated for image", i)

  # =========================
  # PARSE JSON
  # =========================
  try:
    # ANSWERS
    parsed = json.loads(raw_text)
    page_json = {
        "num_page": i,
        "items": parsed
    }
    answers.append(page_json)
  except Exception as e:
    print(f"JSON parsing error on page {i} for full extraction: {e}")

  try:
    # METADATA
    response_metadata["page"]=i
    response_metadata_list.append(response_metadata)
  except Exception as e:
    print(f"JSON metadata parsing error on page {i} for metadata extraction: {e}")

  # =========================
  # SAVE JSON (incrementale)
  # =========================
  with open(JSON_OUTPUT_PATH, "w") as f:
      json.dump(answers, f, indent=2, ensure_ascii=False)

  with open(JSON_METADATA_OUTPUT_PATH, "w") as f:
      json.dump(response_metadata_list, f, indent=2, ensure_ascii=False)

  print(f"Processing completed. JSON saved in {JSON_OUTPUT_PATH}, METADATA JSON saved in {JSON_METADATA_OUTPUT_PATH}")

Working with image: 1
Answer generated for image 1
Processing completed. JSON saved in /content/full_extraction.json, METADATA JSON saved in /content/metadata_full_extraction.json
Working with image: 2
Answer generated for image 2
Processing completed. JSON saved in /content/full_extraction.json, METADATA JSON saved in /content/metadata_full_extraction.json
Working with image: 3
Answer generated for image 3
Processing completed. JSON saved in /content/full_extraction.json, METADATA JSON saved in /content/metadata_full_extraction.json
Working with image: 4
Answer generated for image 4
Processing completed. JSON saved in /content/full_extraction.json, METADATA JSON saved in /content/metadata_full_extraction.json
Working with image: 5
Answer generated for image 5
Processing completed. JSON saved in /content/full_extraction.json, METADATA JSON saved in /content/metadata_full_extraction.json
Working with image: 6
Answer generated for image 6
Processing completed. JSON saved in /content/full

### 3.2) [ALTERNATIVE] Use QWEN-2.5-VL-7B (from Unsloth)
Uncomment the code if you want to use QWEN as local model (requires GPU L4 or more powerful)

In [ ]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth
# else:
#     # Do this only in Colab notebooks! Otherwise use pip install unsloth
#     import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
#     xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
#     !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [ ]:
# from unsloth import FastVisionModel
# import torch

# # Carica il modello pre-addestrato (già in 4-bit per efficienza)
# # Questo è lo stesso modello 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit' usato nel notebook
# model, tokenizer = FastVisionModel.from_pretrained(
#     "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
#     load_in_4bit = True,
# )

# FastVisionModel.for_inference(model) # Abilita la modalità di inferenza

In [ ]:
# def generate_with_qwen():
#     answers=[]
#     for i in range(1,saved_png+1):
#         print("Working with image: ", i)

#         image = Image.open(f"{OUTPUT_PNG_FOLDER}/output{i}.png").convert('RGB')

#         messages = [
#             {"role": "user", "content": [
#                 {"type": "image"},
#                 {"type": "text", "text": SYSTEM_PROMPT}
#             ]}
#         ]

#         input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
#         inputs = tokenizer(
#             image,
#             input_text,
#             add_special_tokens = False,
#             return_tensors = "pt",
#         ).to("cuda")

#         from transformers import TextStreamer
#         text_streamer = TextStreamer(tokenizer, skip_prompt = True)
#         output_ids = model.generate(**inputs, use_cache = True)

#         answer = tokenizer.decode(
#             output_ids[0][inputs["input_ids"].shape[-1]:],
#             skip_special_tokens=True
#         )

#         print("Answer generated for image", i)

#         try:
#             page_json = {"num_page":i, "items":json.loads(answer)}
#             answers.append(page_json)
#         except Exception as e:
#             print(f"Error parsing json in page {i}, trying to recover: {e}")
#             #try stripping ```json at start and ``` at end
#             try:
#                 answer = answer.lstrip("```json").rstrip("```")
#                 page_json = {"num_page":i, "items":json.loads(answer)}
#                 answers.append(page_json)
#             except Exception as e:
#                 print(f"Error parsing json in page {i}, giving up: {e}")

#         # save json
#         with open(JSON_OUTPUT_PATH, 'w') as f:
#             json.dump(answers, f, indent=4)

In [ ]:
# generate_with_qwen()

## 5) Resolve deterministically incomplete questions / answers

In [ ]:
import copy
from typing import List, Dict, Any

NO_QUESTION = "NO_QUESTION"
NO_ANSWER = "NO_ANSWER"


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    return " ".join(str(s).strip().split())


def is_valid_question(item: Dict[str, Any]) -> bool:
    return (
        item.get("num_domanda", -1) != -1
        and normalize_text(item.get("domanda", "")) != NO_QUESTION
    )


def is_orphan_chunk(item: Dict[str, Any]) -> bool:
    """
    Chunk orfano tipico di pagina successiva:
    - num_domanda = -1
    - domanda = NO_QUESTION
    - contiene opzioni
    """
    return (
        item.get("num_domanda", -1) == -1
        and normalize_text(item.get("domanda", "")) == NO_QUESTION
        and len(item.get("opzioni", [])) > 0
    )


def is_incomplete_question(item: Dict[str, Any]) -> bool:
    """
    Assunzione utente:
    le domande complete hanno SEMPRE 4 opzioni.

    Quindi una domanda è incompleta se:
    - è una domanda valida
    - ha meno di 4 opzioni
    """
    return (
        is_valid_question(item)
        and len(item.get("opzioni", [])) < 4
    )


def merge_options(base_options: List[str], extra_options: List[str]) -> List[str]:
    """
    Merge deterministico preservando ordine e rimuovendo duplicati.
    """
    out = []

    for x in base_options + extra_options:
        x = normalize_text(x)
        if x and x not in out:
            out.append(x)

    return out


def repair_cross_page_questions(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Risolve deterministicamente:
    - domande spezzate tra pagine
    - opzioni iniziali della pagina successiva
    - risposta corretta mancante nella pagina precedente
    - rimozione chunk orfani dopo merge

    INPUT:
    [
      {
        "num_page": 1,
        "items": [...]
      }
    ]

    OUTPUT:
    stessa struttura riparata.
    """

    pages = copy.deepcopy(pages)

    for page_idx in range(len(pages) - 1):

        current_items = pages[page_idx]["items"]
        next_items = pages[page_idx + 1]["items"]

        if not current_items or not next_items:
            continue

        last_current = current_items[-1]
        first_next = next_items[0]

        # =========================
        # CASO PRINCIPALE
        # domanda incompleta + chunk orfano successivo
        # =========================
        if is_incomplete_question(last_current) and is_orphan_chunk(first_next):

            current_options = last_current.get("opzioni", [])
            next_options = first_next.get("opzioni", [])

            merged_options = merge_options(current_options, next_options)

            # Tronca a 4 per sicurezza
            merged_options = merged_options[:4]

            last_current["opzioni"] = merged_options

            # =========================
            # FIX RISPOSTA CORRETTA
            # =========================

            current_answer = normalize_text(
                last_current.get("risposta_corretta", NO_ANSWER)
            )

            next_answer = normalize_text(
                first_next.get("risposta_corretta", NO_ANSWER)
            )

            # Se la pagina precedente non aveva risposta
            # ma il chunk successivo sì -> usa quella
            if current_answer == NO_ANSWER and next_answer != NO_ANSWER:
                last_current["risposta_corretta"] = next_answer

            # Se la risposta precedente non è più presente
            # ma quella successiva sì -> sostituisci
            elif (
                current_answer != NO_ANSWER
                and current_answer not in merged_options
                and next_answer in merged_options
            ):
                last_current["risposta_corretta"] = next_answer

            # =========================
            # RIMOZIONE CHUNK ORFANO
            # =========================
            next_items.pop(0)

        # =========================
        # CASO:
        # pagina successiva inizia direttamente
        # con continuazione senza NO_QUESTION
        # =========================
        elif is_incomplete_question(last_current):

            first_next_options = first_next.get("opzioni", [])

            # Se la prima domanda della pagina successiva
            # ha poche opzioni è molto probabilmente
            # continuazione della precedente
            if (
                len(first_next_options) < 4
                and first_next.get("num_domanda") == last_current.get("num_domanda")
            ):

                merged_options = merge_options(
                    last_current.get("opzioni", []),
                    first_next_options
                )

                merged_options = merged_options[:4]

                last_current["opzioni"] = merged_options

                current_answer = normalize_text(
                    last_current.get("risposta_corretta", NO_ANSWER)
                )

                next_answer = normalize_text(
                    first_next.get("risposta_corretta", NO_ANSWER)
                )

                if current_answer == NO_ANSWER and next_answer != NO_ANSWER:
                    last_current["risposta_corretta"] = next_answer

                next_items.pop(0)

    return pages


# =========================================================
# UTILIZZO
# =========================================================

with open(JSON_OUTPUT_PATH, "r", encoding="utf-8") as f:
    raw_pages = json.load(f)

fixed_pages = repair_cross_page_questions(raw_pages)

FIXED_JSON_OUTPUT_PATH = JSON_OUTPUT_PATH.replace(".json", "_fixed.json")

with open(FIXED_JSON_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(fixed_pages, f, indent=2, ensure_ascii=False)

print(f"Fixed JSON salvato in: {FIXED_JSON_OUTPUT_PATH}")

Fixed JSON salvato in: /content/full_extraction_fixed.json


## 4) Data augmentation of the questions using GEMMA-3-27B-IT

In [ ]:
# Limite di 25 chiamate al minuto
sem = asyncio.Semaphore(25)

async def opzioni_augmentation_async(domanda: str, risposta_corretta: str) -> list[str]:
    async with sem:

        # Struttura Few-Shot
        messages = [
            types.Content(role="user", parts=[types.Part(text="Sei un assistente esperto in creazione di test a scelta multipla. Data la domanda e la risposta esatta nel formato DOMANDA | RISPOSTA ESATTA, Genera TRE opzioni errate plausibili ma chiaramente incorrette.")]),
            types.Content(role="user", parts=[types.Part(text="Domanda: 'Molecola bersaglio:' | Risposta corretta: 'DNA'")]),
            types.Content(role="model", parts=[types.Part(text="RNA <SEP> H2O <SEP> CO2")]),
            types.Content(role="user", parts=[types.Part(text="Domanda: 'L’effetto deterministico:' | Risposta corretta: 'È soglia dipendente'")]),
            types.Content(role="model", parts=[types.Part(text="È soglia indipendente <SEP> È soglia randomica <SEP> È soglia deterministica")]),
            # Input attuale
            types.Content(role="user", parts=[types.Part(text=f"Domanda: '{domanda}' | Risposta corretta: '{risposta_corretta}'")])
        ]

        try:
            # Nota: assicurati che l'SDK supporti l'invocazione async (solitamente .aio)
            response = await CLIENT.aio.models.generate_content(
                model="gemma-3-27b-it",
                contents=messages
            )

            text = response.text.strip()
            opzioni_errate = [opt.strip() for opt in text.split("<SEP>")]
            return opzioni_errate
        except Exception as e:
            print(f"Errore durante la chiamata API: {e}")
            return []

async def process_file(full_file_json):
      for p in full_file_json:
        for d in p["items"]:
          domanda = d["domanda"]
          risposta_corretta = d["risposta_corretta"]
          opzioni = d["opzioni"]

          if len(opzioni) == 1 and risposta_corretta == opzioni[0]:
              print(f"Arricchisco domanda {d['num_domanda']}...")

              nuove_opzioni = await opzioni_augmentation_async(domanda, risposta_corretta)

              if len(nuove_opzioni) == 3:
                  opzioni.extend(nuove_opzioni)
                  # Salvataggio incrementale ad ogni successo
                  with open(f"{JSON_OUTPUT_PATH}", "w", encoding="utf-8") as f:
                      json.dump(d, f, indent=2, ensure_ascii=False)
              else:
                  print(f"Errore formato per domanda {d['num_domanda']}")

              # Delay opzionale per gestire meglio il rate limit se necessario
              await asyncio.sleep(3)

In [ ]:
with open(JSON_OUTPUT_PATH, "r", encoding="utf-8") as f:
    full_file_json = json.load(f)

await process_file(full_file_json)

## 5) Separate the json file in different json files based on the section

In [16]:
import json
import os

# Carica il JSON completo
with open("/content/full_extraction_fixed.json", "r", encoding="utf-8") as f:
    full_json = json.load(f)

documents_to_process = []

# Per ogni sezione definita in CONTENT_SECTIONS
for section_info in CONTENT_SECTIONS:
    section_name = section_info["section"]
    section_code = section_info["code"]
    from_page = section_info["from_page"]
    to_page = section_info["to_page"]

    print(f"Processando sezione: {section_name} (pagine {from_page}-{to_page})")

    # Filtra le pagine che appartengono a questa sezione
    lista_domande_risposte = []
    index_domanda = 1
    for page_data in full_json:
        page_num = page_data["num_page"]
        if from_page <= page_num <= to_page:
            for d in page_data["items"]:
                d["num_domanda"] = index_domanda
                lista_domande_risposte.append(d)
                index_domanda += 1

    # Crea il documento in memoria (senza inviare a MongoDB qui)
    if lista_domande_risposte:
        document = {
            "materia": section_name,
            "corso_di_studi": "infermieristica",
            "lista_domande_risposte": lista_domande_risposte
        }
        documents_to_process.append(document)
        print(f"Trovate {len(lista_domande_risposte)} domande per {section_name}")
    else:
        print(f"Nessun dato trovato per la sezione {section_name}")

print("\nSuddivisione completata in memoria!")

Processando sezione: chirurgia (pagine 1-67)
Trovate 656 domande per chirurgia

Suddivisione completata in memoria!


## 6) Validate the file structure and enrich it

In [17]:
import secrets
import string
import random
import re

def clean_option(opt):
    # Rimuove prefissi come "a) ", "b.", "1)", "1 ." all'inizio delle opzioni
    return re.sub(r'^([a-zA-Z]|[0-9]{1,2})[\.\)\-]\s*', '', str(opt).strip()).strip()

def normalize_question(question, source_name):
    required_fields = ["num_domanda", "domanda", "opzioni", "risposta_corretta"]
    for field in required_fields:
        if field not in question:
            raise ValueError(f"Missing required field: {field}")

    if not isinstance(question["opzioni"], list) or len(question["opzioni"]) != 4:
        raise ValueError("'opzioni' must be a list with 4 options")

    # Pulisce le opzioni e la risposta corretta per rimuovere le lettere degli elenchi
    cleaned_options = [clean_option(opt) for opt in question["opzioni"]]
    cleaned_correct_answer = clean_option(question["risposta_corretta"])

    if cleaned_correct_answer not in cleaned_options:
        raise ValueError(f"'risposta_corretta' must be one of the options")

    return {
        "num_domanda": question["num_domanda"],
        "domanda": question["domanda"].strip(),
        "opzioni": cleaned_options,
        "risposta_corretta": cleaned_correct_answer,
        "source_quiz": source_name,
        "cod_domanda": question.get("cod_domanda"),
    }

def process_documents(documents):
    for doc in documents:
        materia = doc["materia"]
        print(f"Validating and enriching: {materia}")

        # Riassegna num_domanda
        for idx, q in enumerate(doc["lista_domande_risposte"], start=1):
            q["num_domanda"] = idx

        # Assegna cod_domanda univoco
        used_codes = set()
        for q in doc["lista_domande_risposte"]:
            if "cod_domanda" in q:
                used_codes.add(q["cod_domanda"])

        for q in doc["lista_domande_risposte"]:
            if "cod_domanda" not in q:
                while True:
                    cod_domanda = "".join(secrets.choice(string.ascii_letters + string.digits) for _ in range(8))
                    if cod_domanda not in used_codes:
                        used_codes.add(cod_domanda)
                        break
                q["cod_domanda"] = cod_domanda

        # Normalizza e valida
        normalized_questions = []
        for idx, q in enumerate(doc["lista_domande_risposte"]):
            try:
                norm = normalize_question(q, materia)
                normalized_questions.append(norm)
            except Exception as e:
                print(f"Error in question {idx + 1} of {materia}: {str(e)}")

        doc["lista_domande_risposte"] = normalized_questions

        # Mescola le opzioni
        for q in doc["lista_domande_risposte"]:
            if "opzioni" in q and isinstance(q["opzioni"], list):
                random.shuffle(q["opzioni"])

    print("Validation and enrichment complete!")

process_documents(documents_to_process)

Validating and enriching: chirurgia
Validation and enrichment complete!


## 7) Save in Mongo DB

In [18]:
from pymongo import MongoClient
import os

db_name = os.getenv("DB_NAME", "quiz_app")
collection_name = os.getenv("COLLECTION_NAME", "infermieristica")

# Connetti a MongoDB
client = MongoClient(MONGO_DB_URI)
db = client[db_name]
collection = db[collection_name]

# Elimina 'chirurgia' per poterla sostituire in modo pulito ed evitare duplicati
res_delete = collection.delete_many({"materia": "chirurgia"})
print(f"Eliminati {res_delete.deleted_count} documenti preesistenti per 'chirurgia'.")

# Inserisci i documenti convalidati e arricchiti
for document in documents_to_process:
    materia = document["materia"]
    result = collection.insert_one(document)
    print(f"Inserito documento aggiornato in MongoDB per la sezione {materia} con ID: {result.inserted_id}")

print("\nSalvataggio su MongoDB completato!")

Eliminati 1 documenti preesistenti per 'chirurgia'.
Inserito documento aggiornato in MongoDB per la sezione chirurgia con ID: 6a0f53f76cff81748d49a5f7

Salvataggio su MongoDB completato!
